# All Pairs

In [1]:
import sys
sys.path.append('..')
from utils.plotting import plot_hic, plot_n_hic, genomic_labels, format_ticks

from notebooks.func_analysis import *
from notebooks.func_plotting import *
from notebooks.func_iou import *
from notebooks.func_pileup import *

import os
import itertools
import pybedtools
from matplotlib_venn import venn2
from collections import defaultdict
from pyBedGraph import BedGraph
import subprocess
import yaml

import bioframe as bf

from io import StringIO

In [2]:
cf = """
samples:

  GM12878_insitu-hic_4DNFI1UEG1HD_25kb:
    file: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_insitu-hic_4DNFI1UEG1HD.hic"
    genome: "hg38"
    norm: "VC_SQRT"
    res: "25000"
    exp: "hic"
    # mcool: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_microc_Wu-2025_GSE225201_hg38.ice.mcool" # Not downloaded but possible
    win: "6000000"
    chroms: "all"
    root_within: "6"
    q_val: "0.5"
    consistency: "0.5"
    length: "8"
    chip_file: '/nfs/turbo/umms-minjilab/downloaded_data/GM12878_Early_repliseq_R1_4DNFI6TILWWX_hg38.bedGraph'
    chrom_size_file: "/nfs/turbo/umms-minjilab/processing/genomes/hg38/hg38.chrom.sizes"
    extra_args:
    - "--diagnostic_plots"

  GM12878_microc_Wu-2025_GSE225201_hg38_25kb:
    file: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_microc_Wu-2025_GSE225201_hg38.hic"
    genome: "hg38"
    norm: "KR"
    res: "25000"
    exp: "hic"
    # mcool: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_microc_Wu-2025_GSE225201_hg38.ice.mcool" # Not downloaded but possible
    win: "6000000"
    chroms: "all"
    root_within: "6"
    q_val: "0.5"
    consistency: "0.5"
    length: "8"
    chip_file: '/nfs/turbo/umms-minjilab/downloaded_data/GM12878_Early_repliseq_R1_4DNFI6TILWWX_hg38.bedGraph'
    chrom_size_file: "/nfs/turbo/umms-minjilab/processing/genomes/hg38/hg38.chrom.sizes"
    extra_args:
    - "--diagnostic_plots"

  GM12878_intacthic_ENCFF318GOM_25kb:
    file: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_intacthic_ENCFF318GOM.hic"
    genome: "hg38"
    norm: "VC_SQRT"
    res: "25000"
    exp: "hic"
    mcool: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_intacthic_ENCFF785BPC.ice.mcool"
    win: "6000000"
    chroms: "all"
    root_within: "6"
    q_val: "0.5"
    consistency: "0.5"
    length: "8"
    chip_file: '/nfs/turbo/umms-minjilab/downloaded_data/GM12878_Early_repliseq_R1_4DNFI6TILWWX_hg38.bedGraph'
    chrom_size_file: "/nfs/turbo/umms-minjilab/processing/genomes/hg38/hg38.chrom.sizes"
    extra_args:
    - "--diagnostic_plots"

  GM12878_RAD21_chiapet_Kim-2024_4DNFIV9RG6YP_hg38_25kb:
    file: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_RAD21_chiapet_Kim-2024_4DNFIV9RG6YP_hg38.hic"
    genome: "hg38"
    norm: "NONE"
    res: "25000"
    exp: "hic"
    mcool: "/nfs/turbo/umms-minjilabs/downloaded_data/GM12878_RAD21_chiapet_Kim-2024_4DNFIV9RG6YP_hg38.ice.mcool"
    win: "6000000"
    chroms: "all"
    root_within: "6"
    q_val: "0.5"
    consistency: "0.5"
    length: "8"
    sum_consistency_im: "True"
    chip_file: '/nfs/turbo/umms-minjilab/downloaded_data/GM12878_Early_repliseq_R1_4DNFI6TILWWX_hg38.bedGraph'
    chrom_size_file: "/nfs/turbo/umms-minjilab/processing/genomes/hg38/hg38.chrom.sizes"
    extra_args:
    - "--diagnostic_plots"

  GM12878_RNAPII_chiapet_Kim-2024_4DNFICWBQKM9_hg38_25kb:
    file: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_RNAPII_chiapet_Kim-2024_4DNFICWBQKM9_hg38.hic"
    genome: "hg38"
    norm: "NONE"
    res: "25000"
    exp: "hic"
    mcool: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_RNAPII_chiapet_Kim-2024_4DNFICWBQKM9_hg38.ice.mcool"
    win: "6000000"
    chroms: "all"
    root_within: "6"
    q_val: "0.5"
    consistency: "0.5"
    length: "8"
    sum_consistency_im: "True"
    chip_file: '/nfs/turbo/umms-minjilab/downloaded_data/GM12878_Early_repliseq_R1_4DNFI6TILWWX_hg38.bedGraph'
    chrom_size_file: "/nfs/turbo/umms-minjilab/processing/genomes/hg38/hg38.chrom.sizes"
    extra_args:
    - "--diagnostic_plots"


  GM12878_RNAPII_chiadrop_Kim-2024_4DNFI3ZH8UYR_hg38_25kb:
    file: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_RNAPII_chiadrop_Kim-2024_4DNFI3ZH8UYR_hg38.hic"
    genome: "hg38"
    norm: "NONE"
    res: "25000"
    exp: "hic"
    mcool: "/nfs/turbo/umms-minjilab/downloaded_data/GM12878_RNAPII_chiadrop_Kim-2024_4DNFI3ZH8UYR_hg38.ice.mcool"
    win: "6000000"
    chroms: "all"
    root_within: "6"
    q_val: "0.5"
    consistency: "0.5"
    length: "8"
    sum_consistency_im: "True"
    chip_file: '/nfs/turbo/umms-minjilab/downloaded_data/GM12878_Early_repliseq_R1_4DNFI6TILWWX_hg38.bedGraph'
    chrom_size_file: "/nfs/turbo/umms-minjilab/processing/genomes/hg38/hg38.chrom.sizes"
    extra_args:
    - "--diagnostic_plots"
"""

config = yaml.safe_load(cf)

samples = config["samples"]

In [3]:
# Basically we want to iterate through each pair and simply plot overlap

In [4]:
names = list(samples.keys())

In [5]:
genome = "hg38"
chromsizes = bf.fetch_chromsizes(db=genome, as_bed=True)
common_chroms = chromsizes["chrom"].tolist()
common_chroms.remove('chrY')

In [6]:
output_name = "v2.0.3"

positions = []
tables = []
hic_files = []
normalizations = []

for s in samples:
    attributes = samples[s]

    hic_file = attributes["file"]
    hic_name = os.path.basename(hic_file).split(".hic")[0]
    resolution = int(attributes["res"])

    # f_s = f"/nfs/turbo/umms-minjilab/sionkim/miajet_output/{s}_chr_combined_50Kb/{s}_chr_combined_{result_type}_50Kb_summary_table.csv"
    # f_e = f"/nfs/turbo/umms-minjilab/sionkim/miajet_output/{s}_chr_combined_50Kb/{s}_chr_combined_{result_type}_50Kb_expanded_table.csv"

    if "q_val" in attributes.keys():
        q_val_str = f"q_val_{attributes['q_val']}"
    else:
        q_val_str = "q_val_0.1" # default 

    if "q_val_white" in attributes.keys():
        q_val_white_str = f"{q_val_str}-{attributes['q_val_white']}"
    elif attributes['exp'] == "replihic" or "compartment" in attributes.keys() and attributes['compartment'] == "False":
        q_val_white_str = f"{q_val_str}" # no white q-value
    else:
        q_val_white_str = f"{q_val_str}-0.95" # default

    folder = f"/nfs/turbo/umms-minjilab/sionkim/miajet_revision/miajet/output_{output_name}/{hic_name}_chr_combined_{genomic_labels(resolution)}"
    prefix = f"{hic_name}_chr_combined_"
    suffix = f"_{genomic_labels(resolution)}_summary_table.csv"
    fname = next(f for f in os.listdir(folder) if f.startswith(prefix) and f.endswith(suffix) and q_val_white_str in f)
    result_type = fname[len(prefix):-len(suffix)]
    f_s = os.path.join(folder, fname)
    f_e = os.path.join(folder, f"{hic_name}_chr_combined_{result_type}_{genomic_labels(resolution)}_expanded_table.csv")


    e = pd.read_csv(f_e, comment="#")
    s = pd.read_csv(f_s, comment="#")

    positions.append(e)
    tables.append(s)
    hic_files.append(hic_file)
    normalizations.append(attributes["norm"])

In [7]:
position_dict = dict(zip(names, positions))
table_dict = dict(zip(names, tables))

In [ ]:
buffer_radius = 3 * resolution  # 3 bins
iou_threshold = 0  # ANY overlap

In [9]:
results = pd.DataFrame(index=names, columns=names, dtype=str)
results_numbers = pd.DataFrame(index=names, columns=names, dtype=float)
unique_identifiers = []
name_pairs = []
jaccard_index_pairs = []
# fill main diagonal with the total number of jets from each method
for n in names:
    results.loc[n, n] = f"{len(table_dict[n])}"
    results_numbers.loc[n, n] = len(table_dict[n])

for n1, n2 in itertools.combinations(names, 2):

    print("-" * 20)
    print(f"Comparing {n1} and {n2}...")

    genome_wide_overlap = 0
    identifiers = []

    for chrom in common_chroms:
        # IOU method is per-chromosome so filter each table by chromosome
        t1 = position_dict[n1].loc[position_dict[n1]["chrom"] == chrom]
        t2 = position_dict[n2].loc[position_dict[n2]["chrom"] == chrom]

        # Summary tables to get the number of jets called by each method
        s1 = table_dict[n1].loc[table_dict[n1]["chrom"] == chrom]
        s2 = table_dict[n2].loc[table_dict[n2]["chrom"] == chrom]

        # Compute the pairs of jets in each direction
        pairs12 = match_by_iou(t1, t2, buffer_radius=buffer_radius, iou_threshold=iou_threshold, x_label="x (bp)", y_label="y (bp)")
        pairs21 = match_by_iou(t2, t1, buffer_radius=buffer_radius, iou_threshold=iou_threshold, x_label="x (bp)", y_label="y (bp)")

        # Construct graph and find unique pairs
        pairs = unique_pairs(pairs12, pairs21, method="optimal")

        # print(f"* {chrom}: {len(pairs)} pairs between {n1} ({len(s1)}) and {n2} ({len(s2)})")

        genome_wide_overlap += len(pairs)
        identifiers += pairs # extend the list

    # Update reuslts table
    len_A = len(table_dict[n1])
    len_B = len(table_dict[n2])
    jacc = genome_wide_overlap / (len_A + len_B - genome_wide_overlap)
    results.loc[n1, n2] = f"{genome_wide_overlap} ({jacc:.3f})"
    results.loc[n2, n1] = f"{genome_wide_overlap} ({jacc:.3f})"  # symmetric

    results_numbers.loc[n1, n2] = genome_wide_overlap
    results_numbers.loc[n2, n1] = genome_wide_overlap  # symmetric

    unique_identifiers.append(identifiers)
    name_pairs.append((n1, n2))
    
print("Genome-wide overlaps:")
results

--------------------
Comparing GM12878_insitu-hic_4DNFI1UEG1HD_25kb and GM12878_microc_Wu-2025_GSE225201_hg38_25kb...
--------------------
Comparing GM12878_insitu-hic_4DNFI1UEG1HD_25kb and GM12878_intacthic_ENCFF318GOM_25kb...
--------------------
Comparing GM12878_insitu-hic_4DNFI1UEG1HD_25kb and GM12878_RAD21_chiapet_Kim-2024_4DNFIV9RG6YP_hg38_25kb...
--------------------
Comparing GM12878_insitu-hic_4DNFI1UEG1HD_25kb and GM12878_RNAPII_chiapet_Kim-2024_4DNFICWBQKM9_hg38_25kb...
--------------------
Comparing GM12878_insitu-hic_4DNFI1UEG1HD_25kb and GM12878_RNAPII_chiadrop_Kim-2024_4DNFI3ZH8UYR_hg38_25kb...
--------------------
Comparing GM12878_microc_Wu-2025_GSE225201_hg38_25kb and GM12878_intacthic_ENCFF318GOM_25kb...
--------------------
Comparing GM12878_microc_Wu-2025_GSE225201_hg38_25kb and GM12878_RAD21_chiapet_Kim-2024_4DNFIV9RG6YP_hg38_25kb...
--------------------
Comparing GM12878_microc_Wu-2025_GSE225201_hg38_25kb and GM12878_RNAPII_chiapet_Kim-2024_4DNFICWBQKM9_hg38_25k

,GM12878_insitu-hic_4DNFI1UEG1HD_25kb,GM12878_microc_Wu-2025_GSE225201_hg38_25kb,GM12878_intacthic_ENCFF318GOM_25kb,GM12878_RAD21_chiapet_Kim-2024_4DNFIV9RG6YP_hg38_25kb,GM12878_RNAPII_chiapet_Kim-2024_4DNFICWBQKM9_hg38_25kb,GM12878_RNAPII_chiadrop_Kim-2024_4DNFI3ZH8UYR_hg38_25kb
GM12878_insitu-hic_4DNFI1UEG1HD_25kb,376,158 (0.261),180 (0.324),41 (0.094),16 (0.038),39 (0.086)
GM12878_microc_Wu-2025_GSE225201_hg38_25kb,158 (0.261),387,183 (0.325),42 (0.095),16 (0.037),37 (0.079)
GM12878_intacthic_ENCFF318GOM_25kb,180 (0.324),183 (0.325),359,33 (0.078),11 (0.027),30 (0.067)
GM12878_RAD21_chiapet_Kim-2024_4DNFIV9RG6YP_hg38_25kb,41 (0.094),42 (0.095),33 (0.078),99,17 (0.121),30 (0.161)
GM12878_RNAPII_chiapet_Kim-2024_4DNFICWBQKM9_hg38_25kb,16 (0.038),16 (0.037),11 (0.027),17 (0.121),58,22 (0.144)
GM12878_RNAPII_chiadrop_Kim-2024_4DNFI3ZH8UYR_hg38_25kb,39 (0.086),37 (0.079),30 (0.067),30 (0.161),22 (0.144),117
